In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.master("local[*]").appName("parquet_to_db").getOrCreate()
base_path = "/home/jovyan/work"

In [3]:
df_anomalies = spark.read.parquet(f"{base_path}/data/traffic_speeds_anomalies/fact")

df_anomalies.printSchema()

root
 |-- day_type: string (nullable = true)
 |-- TIME: string (nullable = true)
 |-- MEDIAN_SPEED: double (nullable = true)
 |-- MAD: double (nullable = true)
 |-- MIN_SPEED_THRESHOLD: double (nullable = true)
 |-- ID: integer (nullable = true)



In [4]:
print(df_anomalies.count())

91008


In [5]:
df_anomalies.write \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://postgres:5432/traffic") \
    .option("dbtable", "anomalies") \
    .option("user", "admin") \
    .option("password", "admin") \
    .option("driver", "org.postgresql.Driver") \
    .mode("overwrite") \
    .save()

In [6]:
df_check = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://postgres:5432/traffic") \
    .option("dbtable", "anomalies") \
    .option("user", "admin") \
    .option("password", "admin") \
    .option("driver", "org.postgresql.Driver") \
    .load()

df_check.show()

+--------+--------+------------+------------------+-------------------+---+
|day_type|    TIME|MEDIAN_SPEED|               MAD|MIN_SPEED_THRESHOLD| ID|
+--------+--------+------------+------------------+-------------------+---+
| weekday|22:30:00|       43.49| 1.240000000000002|  37.05560415122312|395|
| weekday|20:00:00|       42.25|1.8599999999999994|   32.5984062268347|395|
| weekday|18:50:00|       41.01| 2.480000000000004| 28.141208302446234|395|
| weekday|17:55:00|       39.14| 3.730000000000004| 19.784922164566325|395|
| weekday|12:10:00|       35.41| 6.840000000000003|                0.0|395|
| weekday|06:55:00|       36.66| 3.729999999999997|  17.30492216456636|395|
| weekday|06:50:00|       37.28| 3.729999999999997| 17.924922164566365|395|
| weekday|05:05:00|       48.46| 1.240000000000002|  42.02560415122312|395|
| weekday|03:40:00|       47.84|1.2399999999999949|  41.40560415122316|395|
| weekday|03:30:00|       47.84| 1.240000000000002|  41.40560415122312|395|
| weekday|23

In [3]:
df_sensors = spark.read.parquet(f"{base_path}/data/traffic_speeds_anomalies/dim")

df_sensors.printSchema()

root
 |-- ID: integer (nullable = true)
 |-- LINK_POINTS: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- BOROUGH: string (nullable = true)



In [4]:
print(df_sensors.count())

120


In [5]:
df_sensors.write \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://postgres:5432/traffic") \
    .option("dbtable", "sensors") \
    .option("user", "admin") \
    .option("password", "admin") \
    .option("driver", "org.postgresql.Driver") \
    .mode("overwrite") \
    .save()

In [10]:
df_check = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://postgres:5432/traffic") \
    .option("dbtable", "sensors") \
    .option("user", "admin") \
    .option("password", "admin") \
    .option("driver", "org.postgresql.Driver") \
    .load()

df_check.show()

+---+--------------------+--------------------+
| ID|         LINK_POINTS|             BOROUGH|
+---+--------------------+--------------------+
|375|[40.61028,-74.110...|SIE E BRADLEY AVE...|
|184|[40.8347204,-73.8...|               Bronx|
|324|[40.7578106,-73.9...|           Manhattan|
|141|[40.772251,-73.91...|              Queens|
|430|[40.57623,-74.190...|       Staten Island|
|350|[40.63092,-74.145...|       Staten Island|
|215|[40.7369006,-73.9...|           Manhattan|
|354|[40.8859405,-73.8...|               Bronx|
|439|[40.5900505,-74.1...|       Staten Island|
|199|[40.71167,-73.728...|              Queens|
|349|[40.6309004,-74.1...|       Staten Island|
|222|[40.7606904,-73.9...|           Manhattan|
|191|[40.8465405,-73.9...|               Bronx|
|124|[40.68036,-74.004...|           Manhattan|
|149|[40.6916,-73.9991...|           Manhattan|
|190|[40.84671,-73.931...|               Bronx|
|365|[40.741534,-73.95...|              Queens|
|451|[40.7712605,-73.8...|              

In [11]:
df_driving = spark.read.parquet(f"{base_path}/data/driving_behavior")

df_driving.printSchema()

root
 |-- precipitation_type: string (nullable = true)
 |-- intensity: string (nullable = true)
 |-- median_rel: double (nullable = true)



In [12]:
print(df_driving.count())

9


In [13]:
df_driving.write \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://postgres:5432/traffic") \
    .option("dbtable", "driving_behavior") \
    .option("user", "admin") \
    .option("password", "admin") \
    .option("driver", "org.postgresql.Driver") \
    .mode("overwrite") \
    .save()

In [14]:
df_check = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://postgres:5432/traffic") \
    .option("dbtable", "driving_behavior") \
    .option("user", "admin") \
    .option("password", "admin") \
    .option("driver", "org.postgresql.Driver") \
    .load()

df_check.show()

+------------------+---------+------------------+
|precipitation_type|intensity|        median_rel|
+------------------+---------+------------------+
|               dry|     NULL|               1.0|
|              rain|    trace|0.9790444258172674|
|              rain| moderate|0.9135704351281542|
|              snow|    trace| 0.989153254023793|
|              rain|    light| 0.945591322603219|
|              snow|    heavy|0.5131300296484541|
|              rain|    heavy|0.9125262421273618|
|              snow| moderate|0.7976623874305423|
|              snow|    light|0.9080481036077706|
+------------------+---------+------------------+

